# Legacy GUI

저장 모델 예측 노트북을 불러온 뒤 원본 Tkinter GUI를 실행합니다. 화면 구성과 백그라운드 예측 로직은 원본과 동일합니다.


In [ ]:
from pathlib import Path


def find_project_root(start):
    """현재 실행 위치에서 프로젝트 루트를 찾습니다."""
    current = Path(start).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "legacy" / "notebooks" / "최종.ipynb").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾을 수 없습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "legacy" / "전체15.csv"
MODEL_DIR = PROJECT_ROOT / "legacy" / "models"
IMAGE_DIR = PROJECT_ROOT / "legacy" / "images"


In [ ]:
inference_notebook = PROJECT_ROOT / "legacy" / "notebooks" / "03_legacy_saved_model_prediction.ipynb"
get_ipython().run_line_magic("run", str(inference_notebook))


## Tkinter GUI


In [ ]:
def load_image(image_path, size=(100, 100)):
    """이미지를 로드하고 크기를 조정합니다."""
    image = Image.open(image_path)
    image = image.resize(size, Image.LANCZOS)
    return ImageTk.PhotoImage(image)

#####################################################################################################################

# GUI 설정
root = tk.Tk()
root.title("피싱체크_V1")
root.iconbitmap(str(IMAGE_DIR / "chk.ico"))
root.geometry("900x900")


## 상단 바
menubar = tk.Menu(root, bg="darkblue", fg="white", activebackground="lightblue", activeforeground="black")
root.config(menu=menubar)
menubar.add_command(label="HOME", command=lambda: show_frame(main_frame))
menubar.add_command(label="INFO", command=lambda: show_frame(info_frame))
menubar.add_command(label="TEAM", command=lambda: show_frame(team_frame))

# 프레임 생성
main_frame = tk.Frame(root)
team_frame = tk.Frame(root)
info_frame = tk.Frame(root)


def setup_frame(frame):
    for i in range(8):
        frame.grid_rowconfigure(i, weight=1)
    for i in range(3):
        frame.grid_columnconfigure(i, weight=1)

setup_frame(main_frame)
setup_frame(team_frame)
setup_frame(info_frame)

def setup_frame(frame):
    for i in range(8):
        frame.grid_rowconfigure(i, weight=1)
    for i in range(3):
        frame.grid_columnconfigure(i, weight=1)

# 메인 레이아웃
main_frame.grid(row=0, column=0, sticky='nsew')
team_frame.grid(row=0, column=0, sticky='nsew')
info_frame.grid(row=0, column=0, sticky='nsew')

root.grid_rowconfigure(0, weight=1)
root.grid_columnconfigure(0, weight=1)


def create_main_widgets():
    # 타이틀 이미지
    photo0 = load_image(IMAGE_DIR / "타이틀.png", size=(600, 200))
    title_label = tk.Label(main_frame, image=photo0)
    title_label.image = photo0  # 이미지가 사라지지 않도록 참조를 저장
    title_label.grid(row=0, column=0, columnspan=3, pady=20, sticky='n')

    # URL 입력을 위한 내부 프레임 (가운데 정렬에 도움)
    url_frame = tk.Frame(main_frame)
    url_frame.grid(row=1, column=0, columnspan=3, pady=10, sticky='ew')

    # URL 입력 레이블
    tk.Label(url_frame, text="검사할 URL 입력:", font=("Malgun Gothic", 18)).grid(row=0, column=0, padx=20, pady=10, sticky='e')

    # URL 입력 필드 (폰트 크기와 패딩 조정)
    global url_entry
    url_entry = tk.Entry(url_frame, width=40, font=("Malgun Gothic", 18))  # 폰트 크기 증가
    url_entry.grid(row=0, column=1, padx=20, pady=10, sticky='ew')  # 패딩 증가

    # 확인 버튼
    tk.Button(url_frame, text="확인", command=check_url, font=("Malgun Gothic", 18)).grid(row=0, column=2, padx=10, pady=10, sticky='w')

    # url_frame 내부 열 비율 설정 (중앙 정렬을 위해)
    url_frame.grid_columnconfigure(0, weight=1)  # 첫 번째 열 (레이블 열)
    url_frame.grid_columnconfigure(1, weight=1)  # 두 번째 열 (입력 필드 열)
    url_frame.grid_columnconfigure(2, weight=1)  # 세 번째 열 (버튼 열)

    # 결과 프레임
    result_frame = tk.Frame(main_frame, borderwidth=2, relief='groove', bg='white')
    result_frame.grid(row=2, column=0, columnspan=3, pady=10, padx=10, sticky='nsew')

    global result_label
    result_label = tk.Label(result_frame, text="", font=("Malgun Gothic", 14), bg='white')
    result_label.pack(expand=True, fill='both')

    # 특징별 판별 결과
    tk.Label(main_frame, text="▷ 특징별 판별 결과", font=("Malgun Gothic", 19, "bold")).grid(row=4, column=2, pady=5, padx=10, sticky='w')
    tk.Label(main_frame, text="▷ 피싱확률", font=("Malgun Gothic", 19, "bold")).grid(row=4, column=0, pady=5, padx=10, sticky='w')

    # 특징 프레임
    features_frame = tk.Frame(main_frame, borderwidth=2, relief='groove', bg='white')
    features_frame.grid(row=5, column=2, padx=10, pady=10, sticky='nsew')

    global features_text
    features_text = tk.Text(features_frame, font=("Malgun Gothic", 15), bg='white', wrap='word', height=10, width=50)
    features_text.pack(expand=True, fill='both')
    features_text.insert(tk.END, "\n".join([f"{i+1}. {key}: " for i, key in enumerate([
        "IP Address in URL", "URL Length", "Shortening Service", "Having @ Symbol",
        "Double Slash Redirecting", "Prefix/Suffix", "Having Subdomain",
        "Domain Registration Length", "Favicon", "Port", "HTTPS Token",
        "Count Redirection", "Disabling Right Click", "Age of Domain", "DNS Record"])
    ]))
    features_text.config(state=tk.DISABLED)

    # 피싱확률 프레임
    global accuracy_frame
    accuracy_frame = tk.Frame(main_frame, borderwidth=2, relief='groove', bg='white')
    accuracy_frame.grid(row=5, column=0, padx=10, pady=10, sticky='nsew', columnspan=2)

    global accuracy_label
    accuracy_label = tk.Label(accuracy_frame, text=("[비지도 학습]\n""K-means: \n""GMM: \n""MeanShift: \n""Agglomerative: \n\n""\n[지도 학습]\n""Random Forest: \n""Logistic Regression: "), font=("Malgun Gothic", 15), anchor='nw', justify='left', bg='white')
    accuracy_label.grid(row=0, column=0, padx=10, pady=10, sticky='nw')
    
    # 카피라이트 텍스트 추가
    copyright_label = tk.Label(main_frame, text="COPYRIGHT© 2024 JOONGBU UNIVERSITY INFORMATION SECURITY ENGINEERING 4조.ALL RIGHTS RESERVED. ", font=("Malgun Gothic", 10),  bg=main_frame.cget('bg'))
    copyright_label.grid(row=6, column=0, columnspan=3, pady=10, sticky='s')  

def check_url():
    url = url_entry.get()
    if not url:
        messagebox.showerror("오류", "URL을 입력해주세요.")
        return

    if not is_valid_url(url):
        messagebox.showerror("오류", "유효하지 않은 URL 형식입니다.")
        return

    # 진행 상태 팝업창 생성
    show_loading_dialog()

    # 백그라운드에서 URL 검증 수행
    thread = Thread(target=background_check_url, args=(url,))
    thread.start()


# 원형 로딩 애니메이션 클래스 정의
class PulsatingLoading(tk.Canvas):
    def __init__(self, parent, size=100, arc_width=6, base_speed=8, color="green", *args, **kwargs):
        super().__init__(parent, width=size, height=size, *args, **kwargs)
        self.size = size
        self.arc_width = arc_width
        self.base_speed = base_speed
        self.color = color
        self.angle = 0
        self.base_arc_length = 90  # Base length of the arc
        self.speed_variation = 0  # Sticky effect variable
        self.create_arc_shape()
        self.animate()

    def create_arc_shape(self):
        """Creates a pulsating arc shape in the loading spinner."""
        self.delete("all")

        # Pulsating arc length (increases and decreases over time)
        pulsating_arc_length = self.base_arc_length + math.sin(math.radians(self.angle)) * 30  # Varies between 60 and 120 degrees

        self.create_arc(self.arc_width, self.arc_width, self.size - self.arc_width, self.size - self.arc_width,
                        start=self.angle, extent=pulsating_arc_length, outline=self.color, width=self.arc_width, style=tk.ARC)

    def animate(self):
        """Animates the rotating arc with a sticky and pulsating effect."""
        # Sticky effect by modifying speed variation
        self.speed_variation = math.sin(math.radians(self.angle)) * 1.5  # Creates fast-slow-fast effect
        self.angle = (self.angle + self.base_speed + self.speed_variation) % 360  # Adjust the angle dynamically

        self.create_arc_shape()
        self.after(20, self.animate)

# show_loading_dialog 함수 정의
def show_loading_dialog():
    global loading_dialog
    loading_dialog = tk.Toplevel(root)
    loading_dialog.title("처리중")

    dialog_width = 350
    dialog_height = 200

    root_width = root.winfo_width()
    root_height = root.winfo_height()

    root_x = root.winfo_rootx()
    root_y = root.winfo_rooty()

    x = root_x + (root_width - dialog_width) // 2
    y = root_y + (root_height - dialog_height) // 2

    loading_dialog.geometry(f"{dialog_width}x{dialog_height}+{x}+{y}")

    tk.Label(loading_dialog, text="잠시만 기다려 주세요...", font=("Malgun Gothic", 12)).pack(pady=10)

    # 원형 로딩 애니메이션 추가
    pulsating_loading = PulsatingLoading(loading_dialog, size=80, arc_width=6, base_speed=8, color="green")
    pulsating_loading.pack(pady=10)


def update_gui(result, features, kmeans_prob, gmm_prob, meanshift_prob, agglomerative_prob, log_reg_prob, rf_prob):
    try:
        # 확률 값들을 리스트에 저장
        cluster_probs = [kmeans_prob, gmm_prob, meanshift_prob, agglomerative_prob]

        cluster_probs.remove(max(cluster_probs))  # 최고값 제거
        cluster_probs.remove(min(cluster_probs))  # 최저값 제거

        # 남은 2개의 값의 합을 구한 후 2로 나누어 평균 계산
        average_cluster_prob = (cluster_probs[0] + cluster_probs[1]) / 2

        # 결과 텍스트와 색상 결정
        result_text = f"URL: '{url_entry.get()}'\n피싱확률: {average_cluster_prob:.2f}%\n피싱여부: {result}"
        result_label.config(text=result_text, justify='left')

        # result 값이 "정상 사이트"와 일치하는지 확인
        if "정상 사이트" in result:
            result_color = '#0000FF'  # 파란색
        else:
            result_color = 'red'  # 빨간색

        # result_label 업데이트
        result_label.config(text=result_text, fg=result_color, font=("Malgun Gothic", 23, "bold"))

        # features_text 업데이트
        features_text.config(state=tk.NORMAL)

        # 기존 텍스트 삭제
        features_text.delete(1.0, tk.END)
        # 색상 설정
        features_text.tag_configure('black', foreground='black')
        features_text.tag_configure('red', foreground='red')
        features_text.tag_configure('orange', foreground='#FFA500')
        features_text.tag_configure('blue', foreground='blue')

        for i, (key, value) in enumerate(features.items()):
            # Ensure value is a single scalar value
            if isinstance(value, (pd.Series, pd.DataFrame)):
                value = value.iloc[0]  # Extract the first element

            color_tag = 'red' if value == -1 else 'orange' if value == 0 else 'blue'
            features_text.insert(tk.END, f"{i+1}. {key}: ", 'black')
            features_text.insert(tk.END, '피싱' if value == -1 else '정상' if value == 1 else '의심', color_tag)
            features_text.insert(tk.END, '\n')

        features_text.config(state=tk.DISABLED)

        # 정확도 프레임 초기화
        for widget in accuracy_frame.winfo_children():
            widget.destroy()

        # 비지도 학습 구역
        unsupervised_frame = tk.Frame(accuracy_frame, bg=accuracy_frame.cget('bg'))
        unsupervised_frame.pack(fill='x', pady=2, padx=2)

        tk.Label(unsupervised_frame, text="[비지도 학습]", font=("Malgun Gothic", 16, "bold"), bg=unsupervised_frame.cget('bg')).pack(anchor='w', padx=2, pady=2)
        create_bar(unsupervised_frame, "K-means 피싱확률", kmeans_prob)
        create_bar(unsupervised_frame, "GMM 피싱확률", gmm_prob)
        create_bar(unsupervised_frame, "MeanShift 피싱확률", meanshift_prob)
        create_bar(unsupervised_frame, "Agglomerative 피싱확률", agglomerative_prob)
        
        unsupervised_accuracy_label = tk.Label(unsupervised_frame, text=f"비지도학습 평균 피싱확률: {(cluster_probs[0] + cluster_probs[1]) / 2:.2f}%", font=("Malgun Gothic", 15, "bold"), bg=unsupervised_frame.cget('bg'))
        unsupervised_accuracy_label.pack(pady=2, padx=5, anchor='e', side='top')
        
        # 지도 학습 구역
        supervised_frame = tk.Frame(accuracy_frame, bg=accuracy_frame.cget('bg'))
        supervised_frame.pack(fill='x', pady=2, padx=2)

        tk.Label(supervised_frame, text="[지도 학습]", font=("Malgun Gothic", 16, "bold"), bg=supervised_frame.cget('bg')).pack(anchor='w', padx=2, pady=2)
        create_bar(supervised_frame, "Random Forest 피싱확률", rf_prob)
        create_bar(supervised_frame, "Logistic Regression 피싱확률", log_reg_prob)

        supervised_accuracy_label = tk.Label(supervised_frame, text=f"지도학습 평균 피싱확률: {(rf_prob + log_reg_prob) / 2:.2f}%", font=("Malgun Gothic", 15,  "bold"), bg=supervised_frame.cget('bg'))
        supervised_accuracy_label.pack(pady=2, padx=5, anchor='e', side='top')

        # 진행 상태 팝업창 닫기
        loading_dialog.destroy()

    except Exception as e:
        print(f"Error in update_gui: {e}")


def background_check_url(url):
    try:
        # URL을 체크하고 결과를 가져옵니다
        features_df, features_array = feature_extract(url)

        # 반환값이 빈 배열인지 확인합니다
        if features_array.size == 0:
            print("feature_extract 함수에서 유효하지 않은 배열이 반환되었습니다.")
            return

        # 데이터 로드 및 테스트 세트 분할
        data = pd.read_csv(DATA_PATH)
        data = data.drop('url', axis=1, errors='ignore')
        X = data.drop('label', axis=1)
        y = data['label'].values

        # 모델 훈련 시 사용한 feature names를 포함한 DataFrame 생성
        features_df = pd.DataFrame([features_array], columns=X.columns)

        # 군집별 중심값
        kmeans_center = kmeans.cluster_centers_
        gmm_center = gmm.means_
        meanshift_center = meanshift.cluster_centers_
        agglomerative_center = calculate_cluster_representatives(X, cluster_labels)
        
        # 피싱 확률 계산 (클러스터 1에 대한 확률만)
        kmeans_prob = calculate_phishing_probability(features_array, kmeans_center)
        gmm_prob = calculate_phishing_probability(features_array, gmm_center)
        meanshift_prob = calculate_phishing_probability(features_array, meanshift_center)
        agglomerative_prob = calculate_phishing_probability(features_array, agglomerative_center)

        # Logistic Regression과 Random Forest는 predict_proba 메서드를 사용
        log_reg_prob = log_reg.predict_proba([features_array])[0][1] * 100
        rf_prob = rf.predict_proba([features_array])[0][1] * 100

        # 확률 값들을 리스트에 저장
        cluster_probs = [kmeans_prob, gmm_prob, meanshift_prob, agglomerative_prob]

        # 최고값과 최저값을 제거한 나머지 값들
        cluster_probs.remove(max(cluster_probs))  # 최고값 제거
        cluster_probs.remove(min(cluster_probs))  # 최저값 제거

        # 남은 2개의 값의 합을 구한 후 2로 나누어 평균 계산
        average_cluster_prob = (cluster_probs[0] + cluster_probs[1]) / 2

        # GUI 업데이트
        result = f"{'피싱 사이트' if average_cluster_prob > 70 else '정상 사이트'}"
        root.after(0, update_gui, result, features_df, kmeans_prob, gmm_prob, meanshift_prob, agglomerative_prob, log_reg_prob, rf_prob)
        

    except Exception as e:
        print(f"Error during prediction: {e}")

def create_bar(parent_frame, label, value):
    inner_frame = tk.Frame(parent_frame, bg=parent_frame.cget('bg'))
    inner_frame.pack(fill='x', pady=5, padx=5)

    # 라벨에 글씨 크기 15 적용
    label_widget = tk.Label(inner_frame, text=label, font=("Malgun Gothic", 15), width=30, anchor="w", bg=inner_frame.cget('bg'))
    label_widget.pack(side=tk.LEFT, padx=(5, 0))

    bar = ttk.Progressbar(inner_frame, mode='determinate', maximum=100)
    bar['value'] = value
    bar.pack(side=tk.LEFT, fill='x', expand=True, padx=(5, 0))  # fill='x'로 길이 자동 조정

    # 그래프 스타일
    style = ttk.Style()
    style.configure('custom.Horizontal.TProgressbar',
                    troughcolor='white',
                    background='green')
    bar['style'] = 'custom.Horizontal.TProgressbar'
    bar.update_idletasks()

    value_label = tk.Label(inner_frame, text=f"{value:.2f}%", font=("Malgun Gothic", 15), bg=inner_frame.cget('bg'))
    value_label.pack(side=tk.RIGHT, padx=(10, 0))
    
create_main_widgets()

# TEAM 페이지
photo1 = load_image(IMAGE_DIR / "맹구.png")
photo2 = load_image(IMAGE_DIR / "짱구.png")
photo3 = load_image(IMAGE_DIR / "철수.png")
photo4 = load_image(IMAGE_DIR / "유리.png")
photo5 = load_image(IMAGE_DIR / "원장.png")

def create_team_page():
    tk.Label(team_frame, text="담당교수", font=("Malgun Gothic", 20, "bold")).grid(row=0, column=0, columnspan=2, pady=10, padx=10, sticky='ew')
    create_team_member_box(team_frame, photo5, "▷양환석 교수님", "▷역할: 총괄 감독 및 도움", 1, 0)

    tk.Label(team_frame, text="팀원 소개", font=("Malgun Gothic", 20, "bold")).grid(row=2, column=0, columnspan=2, pady=10, padx=10, sticky='ew')

    create_team_member_box(team_frame, photo1, "▷정여진(조장)", "▷학번: 92015441\n▷역할: GMM 클러스터", 3, 0)
    create_team_member_box(team_frame, photo2, "▷양승원", "▷학번: 91913737\n▷역할: K-MEANS 클러스터", 3, 1)
    create_team_member_box(team_frame, photo4, "▷정채영", "▷학번: 92015465\n▷역할: DBSCAN 클러스터", 4, 0)
    create_team_member_box(team_frame, photo3, "▷서장석", "▷학번: 91913543\n▷역할: MEANSHIFT 클러스터", 4, 1)

def create_team_member_box(frame, photo, name, description, row, col):
    member_frame = tk.Frame(frame, borderwidth=2, relief='groove')
    member_frame.grid(row=row, column=col, padx=10, pady=10, sticky='nsew')

    photo_label = tk.Label(member_frame, image=photo)
    photo_label.grid(row=0, column=0, padx=10, pady=5, sticky='nw')

    description_text = f"{name}\n{description}"
    description_label = tk.Label(member_frame, text=description_text, font=("Malgun Gothic", 12), anchor='w', justify='left')
    description_label.grid(row=0, column=1, padx=10, pady=5, sticky='nw')

    member_frame.grid_rowconfigure(0, weight=1)
    member_frame.grid_columnconfigure(1, weight=1)

    frame.grid_columnconfigure(col, weight=1)

create_team_page()

# INFO 페이지
def create_info_page():
    # Canvas와 스크롤바 추가, 높이와 너비 900x900에 맞춤
    canvas = tk.Canvas(info_frame, width=880, height=880)  # 스크롤바를 포함한 크기 조정
    scrollbar = tk.Scrollbar(info_frame, orient="vertical", command=canvas.yview)
    scrollable_frame = tk.Frame(canvas)
    
    scrollable_frame.bind(
        "<Configure>",
        lambda e: canvas.configure(
            scrollregion=canvas.bbox("all")
        )
    )

    canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
    canvas.configure(yscrollcommand=scrollbar.set)

    # 캔버스 및 스크롤바 배치
    canvas.grid(row=0, column=0, sticky='nsew')
    scrollbar.grid(row=0, column=1, sticky='ns')

    info_frame.grid_rowconfigure(0, weight=1)
    info_frame.grid_columnconfigure(0, weight=1)

    # 비지도학습 정보
    tk.Label(scrollable_frame, text="※ 비지도학습 정보", font=("Malgun Gothic", 25, "bold")).grid(row=0, column=0, columnspan=2, pady=10, padx=10, sticky='w')

    # 알고리즘 정보
    create_info_algorithm_box(scrollable_frame, "1) GMM", "(가우시안 혼합 모델)", "▶설명\nGMM은 이름 그대로 가우시안 분포 (정규분포)를\n여러 개 혼합하여 데이터의 복잡한 분포를 근사하기 위한 방법이다.\n\n▶동작과정\n데이터 포인트를 여러 개의 가우시안 분포로 모델링하여 클러스터를 형성합니다.\nEM 알고리즘을 사용하여 클러스터의 평균과 분산을 가중치가 변하지 않을때 까지 반복적으로 업데이트합니다.\n\n※학습률\n- 22,000개의 데이터(정상:15,000개,피싱:7,000개)의 데이터로 학습시켜 까지 나타낸 값", 1, 0, 91.67)
    create_info_algorithm_box(scrollable_frame, "2) K-Means", "(K-평균 군집화)", "▶설명\nK-menas은 클러스터 개수를 미리 정하여 반복적으로 클러스터의 평균을 업데이트 하며\n가장 가까운 점들을 군집화하는 방법\n\n▶동작과정\n데이터를 K개의 클러스터로 나누며, 각 클러스터의 중심까지의 거리의 제곱합을 최소화합니다.\n클러스터 중심을 평균으로 재계산하며 반복합니다.\n\n※학습률\n- 22,000개의 데이터(정상:15,000개,피싱:7,000개)의 데이터로 학습시켜 까지 나타낸 값", 2, 0, 91.72)
    create_info_algorithm_box(scrollable_frame, "3) MeanShift", "(평균 이동 군집화)", "▶설명\nMeanShift 알고리즘은 밀도 기반 클러스터링 알고리즘으로,\n데이터의 밀도가 높은 영역을 찾아 클러스터를 형성하는 방법이다.\n\n▶동작과정\n데이터의 밀도 중심으로 클러스터를 형성합니다.\n클러스터의 중심이 밀도가 가장 높은 지점으로 이동하며, 클러스터의 수와 형태는 데이터 밀도에 따라 결정됩니다.\n\n※학습률\n- 22,000개의 데이터(정상:15,000개,피싱:7,000개)의 데이터로 학습시켜 까지 나타낸 값", 3, 0, 83.26)
    create_info_algorithm_box(scrollable_frame, "4) Agglomerative", "(병합 군집화)", "▶설명\nAgglomerative 알고리즘은 계층적 클러스터링의 한 형태로,\n데이터 포인트를 점진적으로 병합하여 클러스터를 형성하는 방법이다.\n\n▶동작과정\n각 데이터 포인트를 개별 클러스터로 시작한다음,\n가장 가까운 클러스터 쌍을 반복적으로 병합하여 최종 클러스터를 형성한다.\n\n※학습률\n- 22,000개의 데이터(정상:1,5000개,피싱:7,000개)의 데이터로 학습시켜 까지 나타낸 값", 4, 0, 88.37)
    #create_info_algorithm_box(scrollable_frame, "5) DBSCAN", "(밀도 기반 클러스터링)", "▶설명\nDBSCAN은 밀도 기반으로 서로 가까운 데이터 포인트들을 함께 그룹화하는 방법이다.\n\n▶동작과정\n데이터의 밀도를 기반으로 클러스터를 형성하며,\n밀도가 낮은 영역의 데이터 포인트는 잡음으로 인식하며 두 개의 매개변수로 클러스터를 식별합니다.\n\n※학습률\n- 2,2000개의 데이터(정상:15,000개,피싱:7,000개)의 데이터로 학습시켜 {:.2f} 까지 나타낸 값", 5, 0, 68.77)

    # 지도학습 정보 
    tk.Label(scrollable_frame, text="※ 지도학습 정보", font=("Malgun Gothic", 25, "bold")).grid(row=6, column=0, columnspan=2, pady=10, padx=10, sticky='w')

    # 알고리즘 정보
    create_info_algorithm_box(scrollable_frame, "1) RandomForest", "(앙상블 학습)", "▶설명\n다수의 결정트리(Decision Tree)를 조합하여 더 높은 에측성능을 얻는 방법이다.\n\n▶동작과정\n훈련 데이터를 부스트트랩 방식으로 무작위 샘플링하여 여러 데이터셋을 생성한다.\n샘플링된 데이터셋으로 결정 트리를 생성하고 개별적으로 학습한 후 결과를 앙상블 하여 최종 결과를 도출한다.\n\n※※학습률\n- 22,000개의 데이터(정상:1,5000개,피싱:7,000개)의 데이터로 학습시켜 나타낸 값", 7, 0, 96.91)
    create_info_algorithm_box(scrollable_frame, "2) LogisticRegression", "(이진분류모델)", "▶설명\n회귀를 사용하여 데이터가 어떤 범주에 속할 확률을\n0과1로 예측하고 그 확률에 따라 가능성이 더 높은 범주에 속하는 것으로 분류한다.\n\n▶동작과정\n데이터를 준비하여 선형 결합을 만든 값을 시그모이드 함수에 적용 시켜\n0과1사이의 확률로 변환 시킨다.변환된 확률 값이 0.5이상이면 클래스1,0.5미만이면 클래스0으로 분류한다.\n\n※학습률\n- 22,000개의 데이터(정상:15,000개,피싱:7,000개)의 데이터로 학습시켜 나타낸 값", 8, 0, 95.36)


# 알고리즘 정보 박스를 생성
def create_info_algorithm_box(frame, name, name_explanation, description, row, col, percentage):
    algo_frame = tk.Frame(frame, borderwidth=2, relief='groove', padx=10, pady=10)
    algo_frame.grid(row=row, column=col, padx=10, pady=10, sticky='nsew')

    # 큰 글자로 알고리즘 이름과 작게 표시되는 괄호 안의 설명
    name_frame = tk.Frame(algo_frame)
    name_frame.grid(row=0, column=0, padx=10, pady=0, sticky='w')

    title_label = tk.Label(name_frame, text=name, font=("Malgun Gothic", 18, "bold"), anchor='w')
    title_label.pack(side="left")

    explanation_label = tk.Label(name_frame, text=name_explanation, font=("Malgun Gothic", 10), anchor='w')
    explanation_label.pack(side="left")

    # 설명 텍스트
    description_label = tk.Label(algo_frame, text=description, font=("Malgun Gothic", 12), anchor='w', justify='left')
    description_label.grid(row=1, column=0, padx=10, pady=5, sticky='nw')

    # Canvas로 막대 그래프 생성
    canvas = tk.Canvas(algo_frame, width=250, height=30, bg="white", highlightthickness=0)
    canvas.grid(row=2, column=0, padx=10, pady=5, sticky='w')

    # 막대 그래프 배경 (테두리 효과)
    canvas.create_rectangle(5, 5, 245, 25, outline="black", width=2)

    # 퍼센티지에 따른 채워진 막대 그리기
    fill_width = 240 * (percentage / 100)
    canvas.create_rectangle(5, 5, 5 + fill_width, 25, fill="lightblue", outline="")

    # 퍼센트 텍스트 표시
    canvas.create_text(125, 15, text=f"{percentage}%", fill="darkblue", font=("Malgun Gothic", 10, "bold"))

    # 알고리즘 프레임의 크기 비율 조정
    algo_frame.grid_rowconfigure(0, weight=1)
    algo_frame.grid_columnconfigure(0, weight=1)

    # 열 크기 조정
    frame.grid_columnconfigure(col, weight=1)

create_info_page()

def show_frame(frame):
    frame.tkraise()

# 시작 화면
show_frame(main_frame)

root.mainloop()